In [ ]:
# ruff: noqa: F405, F403, E402, F811
# no cache
%load_ext autoreload
%autoreload 2

In [ ]:
# doc-hidden
import os
import sys
import mediapy
import numpy as np

# put the local folder at *the very front*
# The kernel CWD may differ from notebook location, so find src relative to project root
module_path = os.path.abspath("src") if os.path.exists("src/ptspy") else os.path.abspath("../src")

if module_path not in sys.path:
    sys.path.insert(0, module_path)  # use insert(0,…), not append

# throw away the already-imported wheel copy
for m in list(sys.modules):
    if m == "ptspy" or m.startswith("ptspy."):
        del sys.modules[m]

import ptspy  # now comes from ../src

# print(ptspy.__file__)  # sanity-check

In [ ]:
from ptspy import *
from ptspy.more import *
import ipywidgets as widgets

## Quick Start Guide

In [ ]:
@frame(size=(900, 900), bg="fff")
def draw(form):
    form.fillOnly("000").point([450, 450], 30)

draw()

In [ ]:
@frame(bg="000", size=(900, 900))
def draw(form, radius):
    points = regularPolygon(form.size/2, 6, radius)
    form.fillOnly("f0f3f7").points(points, radius/2)

draw(200)

In [ ]:
@frames(bg="fff", size=(900, 900), numFrames=60)
def draw(form, index, radius):
    form.fillOnly("000").point(form.size/2, radius*index)

video = draw(radius=5)
mediapy.show_video(video, codec="h264", fps=30)

In [ ]:
@frames(bg="000", size=(900, 900), numFrames=30)
def draw(form, index, rand):
    row = steps([100, form.height/2 - 50],
                [form.width - 100, form.height/2 - 50], 5)
    row += P(-20, 20) # adjust position
    values = populate(row.shape[0], 1, 1)

    row2 = row + P(0, 120)
    shuffled = np.copy(values)
    rand.shuffle(shuffled)

    form.font(72).fillOnly("fff")
    for pos, val, pos2, val2 in zip(row, values, row2, shuffled):
        form.text(pos, str(int(val))).text(pos2, str(int(val2)))

video = draw(rand=randomizer())
mediapy.show_video(video, codec="h264", fps=10)

In [ ]:

@frame(size=(900, 900))
def draw(form, steps):
    cube = colorCube(steps)
    slice = cube[:, 14, :] # middle chroma slice
    form.image(slice, P([0, 0], form.size))

draw(steps=25)

In [ ]:
@frame(size=(900, 900))
def draw(form, steps):
    cube = colorCube(steps)
    form.fillOnly( colorPicker(cube, 0.20, 0.9, 0.7) )
    form.rect(P([[0,0], form.size]))
    form.fillOnly( colorPicker(cube, 0.18, 0.9, 0.8) )
    form.rect(P([form.size * 0.25, form.size * 0.75]))

draw(30)

In [ ]:
@frame(bg="00000000", size=(900, 900))
def draw(form, cubeSize=24, intensity=0.5, hues=9, swatches=9, variantSeed=0):
    cube = colorCube(steps=cubeSize)
    variant = randomizer(variantSeed)
    palette = colorScheme(cube, hues, swatches, intensity, variant)
    form.colorMap(palette[:, :, 0], P([0, 0], form.size), style="circle")

draw(variantSeed=8)  # random variant with seed 8

In [ ]:
@frame(bg="#00000000", size=(900, 900))
def draw(form):
    cc = colorCube(steps=100)
    # batch 2 interpolations (lightness, chroma, hue) in one call
    startColors = [P(80, 28, 99), P(50, 90, 50)]
    endColors = [P(10, 99, 70), P(80, 30, 10)]
    indices = colorInterpolation(cc, start=startColors,
                                 end=endColors,
                                 steps=8, asPath=True)
    form.colorMap(indices, P([0, 0], form.size))

draw()

In [ ]:
@frame(size=(900, 900))
def draw(form, lightness):
    cc = colorCube(steps=100)

    indices = colorInterpolation(
      cc, P(20, 28, 99), P(50, 99, 10), steps=20, asPath=False
    )

    first = np.squeeze(indices, axis=0)
    form.image(first[:, :, lightness], P([0, 0], form.size))

draw(lightness=5)

In [ ]:
@frame(size=(900, 900))
def draw(form):
    colors = [0xFF2099FF, 0xFFFF66FF, 0x20CCE0FF] # ARGB
    shader = gradient(P([[300,300]]), colors,
                      "sweep", stops=[0.2, 0.5, 0.8])
    form.fill(shader).rect(P([[0, 0], form.size]))

draw()

In [ ]:
@frame(bg="000", size=(900, 900))
def draw(form):
    points = steps([120, 450], [form.width-120, 450],
                   count=20, shaping="cubicOut")
    form.fillOnly("f0f5fa").points(points, 6)

draw()

In [ ]:
# you can skip positional argument or
# use TBD token to set a parameter placeholder (eg, count=TBD)
pointsFn = steps([120, 450], shaping="cubicOut")

@frames(bg="000", size=(900, 900), numFrames=60)
def draw(form, index):
    count = int(index/2.0)
    end = [form.width-40, 40 + index/60.0 * (form.height - 80)]
    form.fillOnly("f0f5fa").points(pointsFn(end, count), 6)

video = draw()
mediapy.show_video(video, codec="h264", fps=30)

In [ ]:
@partial(placeholder=TBD)
def colorDisc(form, color, box):
    form.fillOnly(color).circle(box)

# color palette
cc = colorCube(steps=100)
palette = colorInterpolation(cc, P(0, 28, 99), P(99, 60, 70), steps=90, asPath=True)

# partial example: skipping positional args
first = colorDisc(box=P([30,30], [870, 870]))
# partial example: marking keyword args as TBD
second = colorDisc(form=TBD, color=TBD, box=P([240, 240], [660, 660]))

@frames(bg="000", size=(900, 900), numFrames=90)
def draw(form, index):
    first(form, palette[index])
    second(form=form, color=palette[90-index])

video = draw()
mediapy.show_video(video, codec="h264", fps=30)

In [ ]:
makePolygon = regularPolygon(TBD, sides=4, radius=120, closed=True)

def getNext(form, pts):
    results = []
    for pt in pts:
        next = makePolygon(pt)
        form.strokeOnly("fff", 3).lines(next)
        results.append(next)
    return (form, np.vstack(results))

def makeDot(form, pts):
    form.fillOnly("fff").points(pts, 9)
    return (form, pts)

# same as pipeline(getNext, makeDot, ...)
makeAll = F(getNext) >> makeDot >> getNext

@frame(bg="000", size=(900, 900))
def draw(form):
    poly = makePolygon([450,450])
    form.strokeOnly("fff", 3).lines(poly)
    makeAll(form, poly)

draw()

In [ ]:
makePolygon = regularPolygon(TBD, TBD, radius=90, closed=True)

def getNext(pts, sides):
    if len(pts.shape) > 2:
        pts = pts.reshape(-1, 2)
    return (P([makePolygon(center, sides) for center in pts]), sides)

def makeDot(pts, sides):
    return (pts.reshape(-1, 2), sides)

# get intermediate steps in the pipeline as a generator
makeAll = pipeline(getNext, getNext, getNext, makeDot, lazy=True)
progress = list(makeAll(P(450,450), 6))

@frames(bg="000", size=(900, 900), numFrames=20)
def draw(form, index):
    i = index % len(progress)
    if len(progress[i][0].shape) > 2:
        form.strokeOnly("f0f5fa", 3)
        form.lines(progress[i][0].reshape(-1, 2))
    else:
        form.fillOnly("fff").points(progress[i][0], 6)

video = draw()
mediapy.show_video(video, codec="h264", fps=2)

In [ ]:
@frames(bg="000", size=(900, 900), numFrames=60)
def draw(form, index):
    top = steps(0, form.width, 20,
                shaping=lambda t, a, b: a+(b-a) * (t*t*index/30.0))
    # reshape with new axis and append 0. shape is now (n,2)
    top =  expand(top[:, np.newaxis], 0)
    lines = np.stack([top, top+[0, form.height]], axis=1)

    form.strokeOnly("f0f5fa", 30-index/3.0).lines(lines)

video = draw()
mediapy.show_video(video, codec="h264", fps=30)

In [ ]:
perlin = createPerlinNoise2D([101, 1], seed=123)

@frames(bg="000", size=(1000, 1000), numFrames=100)
def draw(form, t):
    values = perlin(P(t*0.3, t*0.1), 0.4).reshape(-1)
    w, h = form.size
    y = P(0, t/100 * (h+100))
    wave = attachPositionIndex(values) * P(10, 140) + y
    polygon = np.append(wave, P([w, h], [0, h]), axis=0)
    dot = random(shape=3) * [w, h, 5]
    form.fillOnly('fff').polygon(polygon)
    form.points(wave[::5], 30).point(dot, dot[2])


video = draw()
mediapy.show_video(video, codec="h264", fps=30)

In [ ]:
@frame(bg="fff", size=(900, 900))
def draw(form):
    form.fillOnly("000").point(form.size/2, 30)

draw()

In [ ]:
@partial(placeholder=TBD)
def createPolygon(form, center, sides, rotation):
    poly = regularPolygon(center*form.size, radius=form.width/6, sides=sides, closed=True)
    form.save()
    form.rotate(rotation, pivot=center*form.size)
    form.fillOnly("fff", 2).polygon( poly )
    form.restore()

# create 4 polygons in a grid but set size and rotation later
centers = P([0.33, 0.33], [0.66, 0.33], [0.33, 0.66], [0.66, 0.66])
polygons = [createPolygon(form=TBD, center=c, sides=s) for c, s in zip(centers, [4,5,6,7])]

@frames(bg="000", size=(900, 900), numFrames=90)
def draw(form, index):
    [poly(form=form, rotation=index*4) for poly in polygons] # fill in the remaining parameters

video = draw()
mediapy.show_video(video, codec="h264", fps=30)

In [ ]:
makePolygon = regularPolygon(TBD, sides=8, radius=240, closed=True)

def getNext(form, pts):
    results = []
    for pt in pts:
        next = makePolygon(pt)
        form.strokeOnly("fff", 2).lines(next)
        results.append(next)
    return (form, np.vstack(results))

def getNext2(form, pts):
    results = []
    for pt in pts:
        next = makePolygon(pt)
        form.strokeOnly("678", 3).lines(next)
        results.append(next)
    return (form, np.vstack(results))

def makeDot(form, pts):
    form.fillOnly("fff").points(pts, 6)
    return (form, pts)

def makeDot2(form, pts):
    form.fillOnly("000").points(pts, 10)
    return (form, pts)

# same as pipeline(getNext, makeDot, ...)
makeAll = F(getNext2) >> makeDot2 >> getNext >> makeDot


@frame(bg="000", size=(1200, 630))
def draw(form):
    poly = makePolygon([600,315])
    form.strokeOnly("fff", 2).lines(poly)
    makeAll(form, poly)

draw()